# VGGT-Omega JAX Inference Demo

This notebook demonstrates how to load the JAX version of the **VGGT-Omega 1B 512** model and perform inference on real images to predict camera poses, depth maps, and depth confidence. 

The weights have been successfully converted from PyTorch (`.pt`) to Flax Msgpack (`.msgpack.zst`) format.

## 1. Import Dependencies and Setup Backend

In [ ]:
import os
import glob
import time
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Force CPU backend if GPU libraries/drivers are mismatched. 
# Remove or set to "gpu" if JAX CUDA is configured.
os.environ["JAX_PLATFORMS"] = "cpu"

import jax
import jax.numpy as jnp

from vggt_omega.jax.models import VGGTOmega as JAXModel
from vggt_omega.jax.load_weights import load_checkpoint
from vggt_omega.utils.load_fn import load_and_preprocess_images

print(f"JAX Backend: {jax.default_backend().upper()}")
print(f"Devices: {jax.devices()}")

## 1.5 Download Dataset and JAX Model Weights (if not present)

In [ ]:
import urllib.request
import zipfile

def download_file(url, filepath):
    if os.path.exists(filepath):
        print(f"File {filepath} already exists. Skipping download.")
        return
    print(f"Downloading {url} to {filepath}...")
    try:
        req = urllib.request.Request(
            url, 
            headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
        )
        with urllib.request.urlopen(req) as response, open(filepath, 'wb') as out_file:
            chunk_size = 16 * 1024 * 1024  # 16 MB chunks
            while True:
                chunk = response.read(chunk_size)
                if not chunk:
                    break
                out_file.write(chunk)
        print(f"Successfully downloaded {filepath}.")
    except Exception as e:
        if os.path.exists(filepath):
            os.remove(filepath)
        print(f"Error downloading {filepath}: {e}")
        raise e

# Download dataset if not present
if not os.path.exists("nerf_real_360"):
    zip_path = "nerf_real_360.zip"
    download_file(
        "https://huggingface.co/datasets/1kaiser/NERF_360/resolve/main/nerf_real_360.zip?download=true", 
        zip_path
    )
    print("Extracting nerf_real_360.zip...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(".")
    print("Extraction complete.")
else:
    print("Dataset directory nerf_real_360/ already exists.")

# Download JAX weights if not present
download_file(
    "https://huggingface.co/datasets/1kaiser/vggt-omega-jax/resolve/main/vggt_omega_1b_512.msgpack.zst", 
    "vggt_omega_1b_512.msgpack.zst"
)

## 2. Initialize Model and Load Converted Checkpoint

In [ ]:
print("Initializing JAX model...")
jax_model = JAXModel(
    patch_size=16,
    embed_dim=1024,
    enable_camera=True,
    enable_depth=True,
    enable_alignment=False
)

# We will use 2 images for the template shape init: [B, T, H, W, C]
B, T, H, W, C = 1, 2, 512, 512, 3
print("Initializing JAX variable structures...")
variables_template = jax_model.init(jax.random.PRNGKey(0), jnp.zeros((B, T, H, W, C)))

checkpoint_path = "vggt_omega_1b_512.msgpack.zst"
print(f"Restoring weights from: {checkpoint_path}...")
restored_params = load_checkpoint(variables_template, checkpoint_path)
print("Model loaded successfully!")

## 3. Preprocess Input Images
We load real images from the `nerf_real_360/pinecone/images/` directory.

In [ ]:
image_dir = "nerf_real_360/pinecone/images"
image_paths = sorted(glob.glob(os.path.join(image_dir, "*")))[:2]
print(f"Selected Images:")
for p in image_paths:
    print(f"  - {os.path.basename(p)}")

# Load and preprocess to torch shape [T, C, H, W]
x_pt = load_and_preprocess_images(image_paths, image_resolution=512, patch_size=16)
# Add batch dimension -> [B, T, C, H, W]
x_pt = x_pt.unsqueeze(0)

# Convert/Transpose for JAX -> [B, T, H, W, C]
x_jax = jnp.array(x_pt.permute(0, 1, 3, 4, 2).numpy())
print(f"Input shape for JAX: {x_jax.shape}")

## 4. Run Compiled JAX Inference

In [ ]:
@jax.jit
def jax_predict(params, x):
    return jax_model.apply(params, x)

# Cold Run (Compilation + Execution)
print("Running cold run (first run involves JIT compilation)...")
t0 = time.time()
preds = jax_predict(restored_params, x_jax)
# Block until execution is finished to measure execution time properly
for k, v in preds.items():
    if isinstance(v, jnp.ndarray):
        v.block_until_ready()
print(f"Cold run time: {time.time() - t0:.2f} seconds")

# Warm Run (Compiled execution)
print("Running warm run...")
t0 = time.time()
preds = jax_predict(restored_params, x_jax)
for k, v in preds.items():
    if isinstance(v, jnp.ndarray):
        v.block_until_ready()
print(f"Warm run time: {time.time() - t0:.4f} seconds")

## 5. Visualize Predictions

In [ ]:
depth_maps = np.array(preds["depth"])
depth_conf = np.array(preds["depth_conf"])
print(f"Depth shape: {depth_maps.shape}")
print(f"Pose vector shape (B, T, 9): {preds['pose_enc'].shape}")

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for i in range(2):
    # 1. Original Image
    img_np = (x_jax[0, i] * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])).clip(0, 1)
    axes[i, 0].imshow(img_np)
    axes[i, 0].set_title(f"Frame {i} Image")
    axes[i, 0].axis("off")
    
    # 2. Predicted Depth Map
    im1 = axes[i, 1].imshow(depth_maps[0, i, ..., 0], cmap="inferno")
    axes[i, 1].set_title(f"Frame {i} Predicted Depth")
    axes[i, 1].axis("off")
    fig.colorbar(im1, ax=axes[i, 1], fraction=0.046, pad=0.04)
    
    # 3. Predicted Depth Confidence
    im2 = axes[i, 2].imshow(depth_conf[0, i], cmap="viridis")
    axes[i, 2].set_title(f"Frame {i} Depth Confidence")
    axes[i, 2].axis("off")
    fig.colorbar(im2, ax=axes[i, 2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()